In [1]:
from google.colab import files
uploaded = files.upload()

Saving Task 3 and 4_Loan_Data.csv to Task 3 and 4_Loan_Data.csv


In [3]:
import os
print(os.listdir())

['.config', 'Task 3 and 4_Loan_Data.csv', 'sample_data']


In [4]:
import pandas as pd

loans_df = pd.read_csv("Task 3 and 4_Loan_Data.csv")
print(loans_df.head())
print(loans_df.info())

   customer_id  credit_lines_outstanding  loan_amt_outstanding  \
0      8153374                         0           5221.545193   
1      7442532                         5           1958.928726   
2      2256073                         0           3363.009259   
3      4885975                         0           4766.648001   
4      4700614                         1           1345.827718   

   total_debt_outstanding       income  years_employed  fico_score  default  
0             3915.471226  78039.38546               5         605        0  
1             8228.752520  26648.43525               2         572        1  
2             2027.830850  65866.71246               4         602        0  
3             2501.730397  74356.88347               5         612        0  
4             1768.826187  23448.32631               6         631        0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column                 

In [5]:
feature_cols = ['credit_lines_outstanding', 'loan_amt_outstanding',
                 'total_debt_outstanding', 'income', 'years_employed', 'fico_score']

X = loans_df[feature_cols]
y = loans_df['default']

print(X.shape, y.shape)
print(y.value_counts())

(10000, 6) (10000,)
default
0    8149
1    1851
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)

(8000, 6) (2000, 6)


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Scale features - logistic regression works better when features are on similar scales
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

print("Model trained successfully")

Model trained successfully


In [8]:
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]  # probability of default (class 1)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, y_pred_proba))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.999
AUC-ROC: 0.9999883933012768

Confusion Matrix:
[[1630    0]
 [   2  368]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1630
           1       1.00      0.99      1.00       370

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [9]:
recovery_rate = 0.10

def predict_expected_loss(credit_lines_outstanding, loan_amt_outstanding,
                           total_debt_outstanding, income, years_employed, fico_score):

    # Build a single-row dataframe matching training feature order
    input_data = pd.DataFrame([[credit_lines_outstanding, loan_amt_outstanding,
                                 total_debt_outstanding, income, years_employed, fico_score]],
                               columns=feature_cols)

    input_scaled = scaler.transform(input_data)
    pd_estimate = model.predict_proba(input_scaled)[0, 1]  # probability of default

    expected_loss = pd_estimate * (1 - recovery_rate) * loan_amt_outstanding

    return expected_loss, pd_estimate

In [10]:
# Test with a sample borrower
expected_loss, pd_estimate = predict_expected_loss(
    credit_lines_outstanding=5,
    loan_amt_outstanding=10000,
    total_debt_outstanding=15000,
    income=30000,
    years_employed=2,
    fico_score=580
)

print(f"Probability of Default: {pd_estimate:.4f}")
print(f"Expected Loss: ${expected_loss:.2f}")

Probability of Default: 1.0000
Expected Loss: $9000.00


In [11]:
expected_loss, pd_estimate = predict_expected_loss(
    credit_lines_outstanding=0,
    loan_amt_outstanding=5000,
    total_debt_outstanding=3000,
    income=80000,
    years_employed=6,
    fico_score=720
)

print(f"Probability of Default: {pd_estimate:.4f}")
print(f"Expected Loss: ${expected_loss:.2f}")

Probability of Default: 0.0000
Expected Loss: $0.00


In [12]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_model.fit(X_train, y_train)  # trees don't need scaled features

print("Decision tree trained successfully")

Decision tree trained successfully


In [13]:
y_pred_tree = tree_model.predict(X_test)
y_pred_proba_tree = tree_model.predict_proba(X_test)[:, 1]

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_tree))
print("Decision Tree AUC-ROC:", roc_auc_score(y_test, y_pred_proba_tree))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tree))

Decision Tree Accuracy: 0.9965
Decision Tree AUC-ROC: 0.9994055712153872

Confusion Matrix:
[[1627    3]
 [   4  366]]


In [14]:
def predict_expected_loss_comparison(credit_lines_outstanding, loan_amt_outstanding,
                                      total_debt_outstanding, income, years_employed, fico_score):

    input_data = pd.DataFrame([[credit_lines_outstanding, loan_amt_outstanding,
                                 total_debt_outstanding, income, years_employed, fico_score]],
                               columns=feature_cols)

    # Logistic regression (needs scaled input)
    input_scaled = scaler.transform(input_data)
    pd_logistic = model.predict_proba(input_scaled)[0, 1]
    loss_logistic = pd_logistic * (1 - recovery_rate) * loan_amt_outstanding

    # Decision tree (uses raw input)
    pd_tree = tree_model.predict_proba(input_data)[0, 1]
    loss_tree = pd_tree * (1 - recovery_rate) * loan_amt_outstanding

    return {
        "logistic_regression": {"PD": pd_logistic, "expected_loss": loss_logistic},
        "decision_tree": {"PD": pd_tree, "expected_loss": loss_tree}
    }

In [15]:
result = predict_expected_loss_comparison(
    credit_lines_outstanding=5,
    loan_amt_outstanding=10000,
    total_debt_outstanding=15000,
    income=30000,
    years_employed=2,
    fico_score=580
)
print(result)

{'logistic_regression': {'PD': np.float64(0.9999999954690122), 'expected_loss': np.float64(8999.99995922111)}, 'decision_tree': {'PD': np.float64(1.0), 'expected_loss': np.float64(9000.0)}}


In [16]:
fico_data = loans_df[['fico_score', 'default']].copy()
fico_data = fico_data.sort_values('fico_score').reset_index(drop=True)

print(fico_data['fico_score'].min(), fico_data['fico_score'].max())
print(fico_data.shape)

408 850
(10000, 2)


In [17]:
grouped = fico_data.groupby('fico_score')['default'].agg(['count', 'sum']).reset_index()
grouped.columns = ['fico_score', 'total_count', 'default_count']

print(grouped.head())
print(grouped.shape)

   fico_score  total_count  default_count
0         408            1              0
1         409            1              1
2         418            1              1
3         425            1              1
4         438            1              1
(374, 3)


In [18]:
import numpy as np

n_vals = grouped['total_count'].values
k_vals = grouped['default_count'].values
scores = grouped['fico_score'].values
n_unique = len(scores)

def bucket_log_likelihood(n, k):
    if n == 0:
        return 0
    p = k / n
    if p == 0 or p == 1:
        return 0  # perfectly pure bucket contributes 0 to log-likelihood (no uncertainty)
    return k * np.log(p) + (n - k) * np.log(1 - p)

# Precompute cumulative sums for fast range queries
cum_n = np.concatenate([[0], np.cumsum(n_vals)])
cum_k = np.concatenate([[0], np.cumsum(k_vals)])

def range_log_likelihood(i, j):
    # log-likelihood for bucket covering unique-score indices [i, j)
    n = cum_n[j] - cum_n[i]
    k = cum_k[j] - cum_k[i]
    return bucket_log_likelihood(n, k)

print(range_log_likelihood(0, n_unique))  # sanity check: whole dataset as one bucket

-4790.393891227814


In [19]:
def optimal_buckets(num_buckets):
    # dp[b][i] = max log-likelihood using b buckets over the first i unique scores
    # split[b][i] = the best previous cut point for that solution
    NEG_INF = float('-inf')
    dp = [[NEG_INF] * (n_unique + 1) for _ in range(num_buckets + 1)]
    split = [[0] * (n_unique + 1) for _ in range(num_buckets + 1)]

    dp[0][0] = 0

    for b in range(1, num_buckets + 1):
        for i in range(1, n_unique + 1):
            for j in range(b - 1, i):  # j = previous cut point
                if dp[b-1][j] == NEG_INF:
                    continue
                candidate = dp[b-1][j] + range_log_likelihood(j, i)
                if candidate > dp[b][i]:
                    dp[b][i] = candidate
                    split[b][i] = j

    # Reconstruct boundaries
    boundaries_idx = []
    i = n_unique
    b = num_buckets
    while b > 0:
        j = split[b][i]
        boundaries_idx.append(j)
        i = j
        b -= 1
    boundaries_idx.reverse()

    # Convert indices to actual FICO score boundaries
    boundaries = [scores[idx] for idx in boundaries_idx[1:]]  # skip the 0th (start)
    return dp[num_buckets][n_unique], boundaries

best_ll, boundaries = optimal_buckets(5)
print("Best log-likelihood:", best_ll)
print("Bucket boundaries (FICO scores):", boundaries)

Best log-likelihood: -4255.377391458947
Bucket boundaries (FICO scores): [np.int64(521), np.int64(581), np.int64(641), np.int64(697)]


In [20]:
def create_rating_map(boundaries, num_buckets):
    full_boundaries = [min(scores)] + list(boundaries) + [max(scores) + 1]

    def fico_to_rating(fico_score):
        for bucket_idx in range(num_buckets):
            if full_boundaries[bucket_idx] <= fico_score < full_boundaries[bucket_idx + 1]:
                # bucket_idx=0 is lowest scores (worst) -> should get highest rating number
                # bucket_idx=num_buckets-1 is highest scores (best) -> should get rating 1
                return num_buckets - bucket_idx
        return 1  # edge case: max score falls in top bucket

    return fico_to_rating

rating_map = create_rating_map(boundaries, 5)

print("FICO 750 ->", rating_map(750))
print("FICO 450 ->", rating_map(450))
print("FICO 600 ->", rating_map(600))

FICO 750 -> 1
FICO 450 -> 5
FICO 600 -> 3


In [21]:
full_boundaries = [scores.min()] + list(boundaries) + [scores.max() + 1]

for i in range(len(full_boundaries) - 1):
    mask = (fico_data['fico_score'] >= full_boundaries[i]) & (fico_data['fico_score'] < full_boundaries[i+1])
    subset = fico_data[mask]
    default_rate = subset['default'].mean()
    print(f"Bucket {5-i} (FICO {full_boundaries[i]}-{full_boundaries[i+1]-1}): "
          f"n={len(subset)}, default_rate={default_rate:.2%}")

Bucket 5 (FICO 408-520): n=301, default_rate=66.11%
Bucket 4 (FICO 521-580): n=1407, default_rate=38.10%
Bucket 3 (FICO 581-640): n=3438, default_rate=20.45%
Bucket 2 (FICO 641-696): n=3197, default_rate=10.51%
Bucket 1 (FICO 697-850): n=1657, default_rate=4.65%


Using dynamic programming to maximize the log-likelihood of observed defaults, FICO scores were partitioned into 5 buckets with boundaries at approximately 521, 581, 641, and 697. The resulting buckets show a clear, monotonically decreasing default rate (66% to 5%), confirming the boundaries capture meaningful differences in credit risk. This general approach works for any number of buckets and can be re-applied as new data becomes available.